# Reading amr_io metadata and data

This Notebook shows that all metadata items required to read the `method='amr'` data are available.

The data for one snapshot is written into one file, with `nv` sections, containing all data for one variable. Each section consists of sub-sections written by each rank. The patches are written in first-come-first-served order, with offsets given in the patch metadata.  The offset consists of one offset to the start of the variable, another offset for each rank, and finally the offset for the individual patch.

In [1]:
import f90nml
import numpy as np

#### Python, data, and run directories

In [2]:
pwd

'C:\\cygwin64\\home\\codes\\dispatch2\\experiments\\pan\\notebooks'

In [3]:
data='../data/'
run='amr_read_test/'

#### Samples

In [4]:
d=f90nml.read(data+run+'00000/snapshot.nml')
d.keys()

odict_keys(['io_nml', 'snapshot_nml', 'idx_nml', 'time_slices_params', 'cartesian_params'])

In [5]:
io=d['io_nml']
io.keys()

odict_keys(['format', 'ntotal', 'out_time', 'guard_zones', 'time_derivs', 'method'])

In [6]:
sn=d['snapshot_nml']
sn.keys()

odict_keys(['ioformat', 'iout', 'time', 'ntotal', 'istep', 'mesh_type', 'position', 'size', 'ds', 'box', 'velocity', 'level', 'quality', 'gamma', 'ncell', 'li', 'ui', 'n', 'ng', 'gn', 'nv', 'nt', 'nw', 'kind', 'eos', 'opacity', 'periodic', 'guard_zones', 'time_derivs', 'no_mans_land'])

In [7]:
d=f90nml.read(data+run+'00000/rank_00000_patches.nml')
d.keys()

odict_keys(['idx_nml', 'patch_nml', 'offset_nml'])

In [8]:
of=d['offset_nml']
of.keys()

odict_keys(['rank_offset', 'var_size'])

In [9]:
pp=d['patch_nml']
pa=pp[0]
pa.keys()

odict_keys(['id', 'time', 'position', 'size', 'level', 'dtime', 'times', 'dtimes', 'task_offset'])

In [10]:
if sn['guard_zones']:
    shape=sn['gn']
else:
    shape=sn['n']
shape=tuple(shape)
iv=1
offset=pa['task_offset']+of['rank_offset']+iv*of['var_size']
shape,offset

((20, 20, 20), 1040000)

### Template procedure

In [11]:
it=1        # snapshot
iv=4        # variable
ra=0        # rank
ip=0        # patch
sd='{:05d}/'.format(it)
file=data+run+sd+'snapshot.nml'
d=f90nml.read(file); print(file)
sn=d['snapshot_nml']
if sn['guard_zones']:
    shape=sn['gn']
else:
    shape=sn['n']
file=data+run+sd+'rank_{:05d}_patches.nml'.format(ra)
d=f90nml.read(file); print(file)
pp=d['patch_nml']
pa=pp[ip]
shape=tuple(shape)
file=data+run+sd+'snapshot.dat'
fo=open(file,'rb'); print(file)
of=d['offset_nml']
print('patch:',ip,' variable:',iv)
offset=pa['task_offset']+of['rank_offset']+iv*of['var_size']
mm=np.memmap(fo,dtype=np.float32,offset=4*offset,mode='r',
             order='F',shape=shape)
(mm.min(),mm.max())

../data/tmp/00003/snapshot.nml
../data/tmp/00003/rank_00000_patches.nml
../data/tmp/00003/snapshot.dat
patch: 0  variable: 4


(memmap(0.911578357219696, dtype=float32),
 memmap(1.1658122539520264, dtype=float32))